12-22-25
Experimenting with running updated version of cwe_db with simhashing


In [ ]:
import shutil
from cwe_db import CVE_DB

# Copy the source database to a new file

source_db = 'datasets/bugsinpy.db'
output_db = 'datasets/bugsinpy_simhashk=5.db'

print(f"Copying {source_db} to {output_db}...")
shutil.copy(source_db, output_db)

# Open the new database and apply simhashing
print("Applying simhashing to remove near-duplicates...")
db = CVE_DB(output_db)

# Performance optimizations
db.cur.execute("PRAGMA optimize")
db.cur.execute("PRAGMA journal_mode = WAL")

# Get count before simhashing
before_count = db.cur.execute("SELECT COUNT(*) FROM funcs").fetchone()[0]
print(f"Functions before simhashing: {before_count}")

# Apply simhashing with k=5 (removes functions with Hamming distance <= 5)
db.simhash(k=5)

# Get count after simhashing
after_count = db.cur.execute("SELECT COUNT(*) FROM funcs").fetchone()[0]
print(f"Functions after simhashing: {after_count}")
print(f"Removed {before_count - after_count} near-duplicate functions")

db.close()
print(f"\nOutput saved to: {output_db}")

Copying datasets/bugsinpy.db to datasets/bugsinpy_simhashk=5.db...
Applying simhashing to remove near-duplicates...
Functions before simhashing: 39858
Functions after simhashing: 13963
Removed 25895 near-duplicate functions

Output saved to: datasets/bugsinpy_simhashk=5.db
Functions after simhashing: 13963
Removed 25895 near-duplicate functions

Output saved to: datasets/bugsinpy_simhashk=5.db


12-30-25

After manually simhashing with threshold k=3 and k=5 for hamming distance, going to automate for k=4, k=5 up to k = 64.



In [ ]:
import shutil
import time
import csv
import os
from datetime import datetime
from cwe_db import CVE_DB

# Configuration

source_dbs = [
    'datasets/bugsinpy.db',
    'datasets/devign.db', 
    'datasets/juliet_c.db',
    'datasets/juliet_java.db',
    'datasets/juliet_csharp.db'
]
output_dir = 'simhash_datasets/'
log_file = 'simhash_log.csv'

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

# Create or append to log file
log_exists = os.path.exists(log_file)
with open(log_file, 'a', newline='') as csvfile:
    fieldnames = ['timestamp', 'source_db', 'k_value', 'before_count', 'after_count', 
                  'removed_count', 'removal_percentage', 'execution_time_seconds']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    
    if not log_exists:
        writer.writeheader()
    
    # Process each source database
    for source_db in source_dbs:
        db_name = os.path.basename(source_db).replace('.db', '')
        print(f"\n{'='*60}")
        print(f"Processing: {db_name}")
        print(f"{'='*60}")
        
        # Get initial count from source database
        temp_db = CVE_DB(source_db)
        initial_count = temp_db.cur.execute("SELECT COUNT(*) FROM funcs").fetchone()[0]
        temp_db.close()
        
        # Process k values 1 to 15
        for k in range(1, 16):
            # if k in skip_numbers:
            #     print(f"Skipping k={k} (already processed)")
            #     continue
            
            # Create output database name
            output_db = os.path.join(output_dir, f"{db_name}_simhash_k={k}.db")
            
            # Skip if already exists
            if os.path.exists(output_db):
                print(f"Output database {output_db} already exists, skipping...")
                continue
            
            print(f"\nProcessing k={k}...")
            print(f"Copying {source_db} to {output_db}...")
            
            # Copy source database
            shutil.copy(source_db, output_db)
            
            # Open the new database
            db = CVE_DB(output_db)
            
            # Apply performance optimizations
            db.cur.execute("PRAGMA optimize")
            db.cur.execute("PRAGMA journal_mode = WAL")
            
            # Get count before simhashing
            before_count = db.cur.execute("SELECT COUNT(*) FROM funcs").fetchone()[0]
            
            # Apply simhashing with current k value and time it
            start_time = time.time()
            db.simhash(k=k)
            execution_time = time.time() - start_time
            
            # Get count after simhashing
            after_count = db.cur.execute("SELECT COUNT(*) FROM funcs").fetchone()[0]
            removed_count = before_count - after_count
            removal_percentage = (removed_count / before_count * 100) if before_count > 0 else 0
            
            # Close database
            db.close()
            
            # Log results
            log_entry = {
                'timestamp': datetime.now().isoformat(),
                'source_db': db_name,
                'k_value': k,
                'before_count': before_count,
                'after_count': after_count,
                'removed_count': removed_count,
                'removal_percentage': round(removal_percentage, 2),
                'execution_time_seconds': round(execution_time, 2)
            }
            writer.writerow(log_entry)
            
            print(f"  Functions before simhashing: {before_count}")
            print(f"  Functions after simhashing (k={k}): {after_count}")
            print(f"  Removed {removed_count} near-duplicate functions ({removal_percentage:.2f}%)")
            print(f"  Execution time: {execution_time:.2f} seconds")
            print(f"  Output saved to: {output_db}")

print(f"\n{'='*60}")
print("All processing complete!")
print(f"Log saved to: {log_file}")
print(f"Output databases saved to: {output_dir}")

# Print summary from log file
print(f"\n{'='*60}")
print("SUMMARY OF RESULTS:")
print(f"{'='*60}")

with open(log_file, 'r') as csvfile:
    reader = csv.DictReader(csvfile)
    rows = list(reader)
    
    # Group by database
    db_groups = {}
    for row in rows:
        db_name = row['source_db']
        if db_name not in db_groups:
            db_groups[db_name] = []
        db_groups[db_name].append(row)
    
    # Print summary for each database
    for db_name, entries in db_groups.items():
        print(f"\n{db_name}:")
        print("-" * 40)
        print(f"{'k':<5} {'Before':<8} {'After':<8} {'Removed':<8} {'Removal %':<10} {'Time(s)':<8}")
        print("-" * 40)
        
        for entry in sorted(entries, key=lambda x: int(x['k_value'])):
            print(f"{entry['k_value']:<5} "
                  f"{entry['before_count']:<8} "
                  f"{entry['after_count']:<8} "
                  f"{entry['removed_count']:<8} "
                  f"{entry['removal_percentage']:<10} "
                  f"{entry['execution_time_seconds']:<8}")
        
        # Calculate totals
        if entries:
            initial_count = entries[0]['before_count']  # First k value should have initial count
            print(f"\n  Initial functions in source: {initial_count}")
            print(f"  Most aggressive (k=15): {entries[-1]['after_count']} functions remaining")
            
            # Show reduction from most aggressive k
            final_entry = max(entries, key=lambda x: int(x['k_value']))
            total_reduction = int(initial_count) - int(final_entry['after_count'])
            reduction_pct = (total_reduction / int(initial_count) * 100) if int(initial_count) > 0 else 0
            print(f"  Total reduction across all k values: {total_reduction} functions ({reduction_pct:.1f}%)")

print(f"\n{'='*60}")

print(f"{'='*60}")


Processing: bugsinpy

Processing k=1...
Copying datasets/bugsinpy.db to simhash_datasets/bugsinpy_simhash_k=1.db...
  Functions before simhashing: 39858
  Functions after simhashing (k=1): 15945
  Removed 23913 near-duplicate functions (60.00%)
  Execution time: 21.46 seconds
  Output saved to: simhash_datasets/bugsinpy_simhash_k=1.db

Processing k=2...
Copying datasets/bugsinpy.db to simhash_datasets/bugsinpy_simhash_k=2.db...
  Functions before simhashing: 39858
  Functions after simhashing (k=2): 15427
  Removed 24431 near-duplicate functions (61.30%)
  Execution time: 21.32 seconds
  Output saved to: simhash_datasets/bugsinpy_simhash_k=2.db
Skipping k=3 (already processed)

Processing k=4...
Copying datasets/bugsinpy.db to simhash_datasets/bugsinpy_simhash_k=4.db...
  Functions before simhashing: 39858
  Functions after simhashing (k=4): 14479
  Removed 25379 near-duplicate functions (63.67%)
  Execution time: 20.63 seconds
  Output saved to: simhash_datasets/bugsinpy_simhash_k=4.